# Profiling LLMeter Runs

This notebook demonstrates the `ProfileCallback` — a built-in callback that
collects per-request timing data and generates a profiling report.

## What you'll learn

1. How to attach the profiler to a Runner
2. Reading the profile report (time accounting, cache hits, throughput)
3. Inspecting per-invocation profiles
4. Generating profiling plots (phase breakdown, timeline, throughput)
5. Loading saved profiles from disk for post-hoc analysis

In [ ]:
from llmeter.callbacks import ProfileCallback
from llmeter.endpoints import OpenAIResponseStreamEndpoint
from llmeter.runner import Runner

## 1. Setup

Create an endpoint and attach the `ProfileCallback` to a Runner.
The profiler works with any endpoint — streaming endpoints give the richest data.

Here we use the OpenAI Responses API on Bedrock Mantle with GPT-5.4.

In [ ]:
import os
from aws_bedrock_token_generator import provide_token

AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
MODEL_ID = "openai.gpt-5.4"

token = provide_token(region=AWS_REGION)

endpoint = OpenAIResponseStreamEndpoint(
    model_id=MODEL_ID,
    endpoint_name="GPT-5.4-mantle",
    provider="bedrock-mantle",
    api_key=token,
    base_url=f"https://bedrock-mantle.{AWS_REGION}.api.aws/openai/v1",
)

profiler = ProfileCallback(
    print_report=True,   # Print summary when the run finishes
    save_report=True,    # Save profile_report.json + profile_invocations.jsonl
)

runner = Runner(
    endpoint=endpoint,
    output_path="./outputs/profiling-demo",
    callbacks=[profiler],
)

## 2. Run the Benchmark

Run a small batch. The profiler will automatically collect per-request
timing data and produce a report at the end.

In [ ]:
payload = endpoint.create_payload(
    "Explain the difference between amortized and average-case analysis "
    "in algorithm design. Give one example of each. Keep it under 150 words."
)

result = await runner.run(
    payload=payload,
    n_requests=10,
    clients=2,
    run_name="profiling-demo",
)

## 3. Inspect the Report

The report is printed automatically (since `print_report=True`), but
you can also access it programmatically.

In [ ]:
report = profiler.report

print(f"Wall clock:      {report.total_wall_clock:.1f}s")
print(f"API time:        {report.total_api_time:.1f}s ({report.api_time_fraction:.0%})")
print(f"Runner overhead: {report.runner_overhead:.1f}s ({report.runner_overhead_fraction:.0%})")
print(f"Cache hit rate:  {report.cache_hit_rate:.0%}")
print(f"Total retries:   {report.total_retries}")
print(f"Tokens in/out:   {report.total_tokens_input:,} / {report.total_tokens_output:,}")

## 4. Per-Invocation Data

Each request's profile is available as an `InvocationProfile` object.
This is the same data saved to `profile_invocations.jsonl`.

In [ ]:
import pandas as pd

# Convert to DataFrame for easy analysis
from dataclasses import asdict

df = pd.DataFrame([asdict(p) for p in profiler.invocation_profiles])
df[["sequence", "ttft", "generation_time", "output_tokens_per_second",
    "tokens_input", "tokens_output", "cache_hit", "error"]]

## 5. Profiling Plots

The `llmeter.plotting` module provides ready-made visualizations for profiling data.

In [ ]:
from llmeter.plotting import (
    plot_phase_breakdown,
    plot_request_timeline,
    plot_throughput_over_time,
    plot_time_accounting,
    plot_tpot_distribution,
    plot_ttft_vs_input_tokens,
    plot_profile_summary,
)

### Phase Breakdown

Stacked bar showing TTFT vs generation time for each request.

In [ ]:
plot_phase_breakdown(profiler).show()

### Request Timeline

Gantt-style view showing request concurrency. Overlapping bars = parallel requests.

In [ ]:
plot_request_timeline(profiler).show()

### Throughput Over Time

How token generation speed varies as the run progresses.

In [ ]:
plot_throughput_over_time(profiler, window=3).show()

### Time Accounting

Where does wall-clock time go? TTFT (prefill), stream (decode), or runner overhead?

In [ ]:
plot_time_accounting(profiler).show()

### TPOT Distribution

Histogram of per-token latency across all requests.

In [ ]:
plot_tpot_distribution(profiler).show()

### TTFT vs Input Tokens

Does prefill time scale with prompt length? Cache hits are highlighted.

In [ ]:
plot_ttft_vs_input_tokens(profiler).show()

### All-in-One Summary

A 2×2 multi-panel combining phase breakdown, throughput, TPOT, and time accounting.

In [ ]:
plot_profile_summary(profiler).show()

## 6. Loading Saved Profiles

Profile data is saved alongside the run results. You can reload it later
without needing the original callback or live endpoint.

In [ ]:
import json
from pathlib import Path
from llmeter.callbacks.profiling import InvocationProfile

# Load per-invocation profiles from saved JSONL
profile_path = Path("./outputs/profiling-demo/profiling-demo/profile_invocations.jsonl")

if profile_path.exists():
    with open(profile_path) as f:
        profiles = [InvocationProfile(**json.loads(line)) for line in f]
    
    print(f"Loaded {len(profiles)} invocation profiles")
    
    # These can be passed directly to plotting functions
    plot_phase_breakdown(profiles, title="Loaded from disk").show()
else:
    print(f"No saved profiles found at {profile_path}")
    print("(Run the benchmark cells above first)")

In [ ]:
# Load the summary report
report_path = Path("./outputs/profiling-demo/profiling-demo/profile_report.json")

if report_path.exists():
    with open(report_path) as f:
        report_data = json.load(f)
    
    print("Saved profile report:")
    for key, value in report_data.items():
        if not isinstance(value, dict):
            print(f"  {key}: {value}")
else:
    print(f"No saved report found at {report_path}")

## 7. Combining with Other Callbacks

The profiler plays well with other callbacks like `SystemMetricsMonitor` and `CostModel`.

In [ ]:
from llmeter.callbacks import CostModel, ProfileCallback
from llmeter.callbacks.system_metrics import SystemMetricsMonitor
from llmeter.callbacks.cost.dimensions import InputTokens, OutputTokens

# Combine profiling with cost estimation and system monitoring
profiler = ProfileCallback()
monitor = SystemMetricsMonitor(sample_interval=0.5)
cost_model = CostModel(
    request_dims=[
        InputTokens(price_per_million=3.0),
        OutputTokens(price_per_million=15.0),
    ]
)

runner = Runner(
    endpoint=endpoint,
    output_path="./outputs/profiling-full",
    callbacks=[profiler, monitor, cost_model],
)

result = await runner.run(payload=payload, n_requests=10, run_name="full-profile")
print("Uncomment the line above to run with all callbacks")